# Lab 16: Investigating a Cloud Account with CloudTrail Logs
Course 314341 AI in Cybersecurity and Cloud Computing, Ajloun National University. Dr. Mohammed Tawfik.

Run the cells from top to bottom in Google Colab (Runtime → Run all, or Shift+Enter cell by cell). Read the comments: lines that start with # explain the code.

**Data:** `cloudtrail_events.json`, a small CloudTrail-style log of a university cloud account made for this course (values invented, field names as in AWS CloudTrail). Loaded from the course GitHub: nothing to upload. Something happened on the night of 3 November. Find it.

## 1. Load the events

In [ ]:
import pandas as pd
url = "https://raw.githubusercontent.com/kmkholm/ai-cybersecurity-cloud-course/main/labs/data/cloudtrail_events.json"
ev = pd.read_json(url)
ev["eventTime"] = pd.to_datetime(ev["eventTime"])
print(ev.shape)
ev.head()

## 2. What is normal for this account?

In [ ]:
print(ev["userName"].value_counts(), "\n")
print(ev["sourceIPAddress"].value_counts(), "\n")
print(ev["awsRegion"].value_counts())

## 3. Events from an unusual address or region

In [ ]:
usual_ips = ["198.51.100.20"]           # the university address
usual_regions = ["eu-central-1"]        # the region the university uses
odd = ev[~ev["sourceIPAddress"].isin(usual_ips) | ~ev["awsRegion"].isin(usual_regions)]
odd.sort_values("eventTime")[["eventTime", "userName", "eventName", "sourceIPAddress", "awsRegion", "resource"]]

## 4. What did the attacker do? Count and timeline

In [ ]:
print(odd["eventName"].value_counts())
print("from", odd["eventTime"].min(), "to", odd["eventTime"].max())

## 5. Is every night event an attack?

In [ ]:
night = ev[ev["eventTime"].dt.hour.isin([21, 22, 23, 0, 1, 2, 3])]
night.groupby(["userName", "sourceIPAddress", "eventName"]).size()